# 09 — Pre-training, fine-tuning, PEFT/LoRA, prompt tuning, and post-training

## Mental model

Modern NLP model lifecycle:

```text
pre-training → instruction tuning/SFT → preference optimization/post-training → product adaptation
```

Adaptation options:

- prompting: no weight update
- full fine-tuning: update all or many weights
- PEFT/LoRA: train a small number of parameters
- RAG: adapt knowledge without changing weights
- post-training: shape behavior using demonstrations, preferences, rewards, or verifiers

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

## Toy LoRA from scratch

LoRA freezes a base weight matrix and learns a low-rank update:

```text
W' = W + BA
```

This cell implements a LoRA linear layer so you understand what PEFT is doing under the hood.

In [ ]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # freeze base weights
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        lora_update = (x @ self.A.t()) @ self.B.t()
        return self.base(x) + self.scaling * lora_update

layer = LoRALinear(10, 3, rank=2)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())
print({"trainable": trainable, "total": total, "pct_trainable": trainable/total})

## Toy task: adapt a frozen model with LoRA

This is deliberately small, but the same idea scales to attention projection matrices in LLMs.

In [ ]:
# Synthetic binary classification dataset
rng = np.random.default_rng(SEED)
X = torch.tensor(rng.normal(size=(300, 10)), dtype=torch.float32)
true_w = torch.randn(10, 2)
y = (X @ true_w).argmax(dim=1)

model = nn.Sequential(
    LoRALinear(10, 16, rank=2, alpha=4.0),
    nn.ReLU(),
    LoRALinear(16, 2, rank=2, alpha=4.0),
)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.05)

for epoch in range(30):
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

acc = (model(X).argmax(dim=1) == y).float().mean().item()
print("training accuracy", acc)

## Hugging Face PEFT LoRA template

Use this for real model adaptation when you have domain examples and want lower cost than full fine-tuning.

In [ ]:
peft_lora_template = r"""
# %pip install transformers peft accelerate datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

model_name = "gpt2"  # replace with your base model
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],  # depends on architecture
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

# Prepare dataset with fields such as {"text": "Instruction... Response..."}
# Tokenize, then train with Trainer or SFTTrainer.
"""
print(peft_lora_template)

## Prompt tuning concept

Prompt tuning freezes the model and learns soft prompt embeddings prepended to the input. Unlike hard prompts, these learned vectors are not human-readable.

In [ ]:
class SoftPromptWrapper(nn.Module):
    """Conceptual soft-prompt wrapper around an embedding layer."""
    def __init__(self, embedding_layer, prompt_len=10):
        super().__init__()
        self.embedding_layer = embedding_layer
        emb_dim = embedding_layer.embedding_dim
        self.soft_prompt = nn.Parameter(torch.randn(prompt_len, emb_dim) * 0.02)

    def forward(self, input_ids):
        batch_size = input_ids.size(0)
        token_embs = self.embedding_layer(input_ids)
        prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        return torch.cat([prompt, token_embs], dim=1)

emb = nn.Embedding(100, 32)
wrapper = SoftPromptWrapper(emb, prompt_len=5)
input_ids = torch.randint(0, 100, (2, 7))
print(wrapper(input_ids).shape)  # batch, prompt+tokens, dim

## SFT data format

Supervised fine-tuning teaches the model the desired response style and format.

In [ ]:
sft_example = {
    "instruction": "Extract the contract risk from this clause.",
    "input": "The supplier may terminate the agreement at any time without notice.",
    "output": {
        "risk_type": "termination",
        "severity": "high",
        "rationale": "The supplier has unilateral termination rights without notice."
    }
}
sft_example

## DPO loss from scratch

DPO uses chosen/rejected preference pairs. The core idea is to increase the policy's relative preference for chosen responses compared with a reference model.

This is a toy scalar implementation for learning the formula.

In [ ]:
def dpo_loss(policy_chosen_logp, policy_rejected_logp, ref_chosen_logp, ref_rejected_logp, beta=0.1):
    policy_logratio = policy_chosen_logp - policy_rejected_logp
    ref_logratio = ref_chosen_logp - ref_rejected_logp
    logits = beta * (policy_logratio - ref_logratio)
    return -F.logsigmoid(logits).mean()

policy_chosen = torch.tensor([-2.0, -1.0, -0.5])
policy_rejected = torch.tensor([-3.0, -2.0, -1.5])
ref_chosen = torch.tensor([-2.5, -1.2, -0.7])
ref_rejected = torch.tensor([-2.8, -1.8, -1.2])
print(dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected))

## Programmatic verifiers

For code, math, SQL, extraction, and workflow tasks, use automatic checks as rewards or evals.

In [ ]:
def verify_json_keys(output, required_keys):
    return isinstance(output, dict) and all(k in output for k in required_keys)

outputs = [
    {"risk_type": "termination", "severity": "high", "rationale": "..."},
    {"risk": "termination"},
]
for o in outputs:
    print(o, verify_json_keys(o, ["risk_type", "severity", "rationale"]))

## Product adaptation rule of thumb

| Need | Best first move |
|---|---|
| New knowledge | RAG |
| Better format/style | Prompting or SFT |
| Cheaper/smaller task model | Fine-tune/LoRA |
| Better safety/tone | Post-training + evals |
| Better factuality | Retrieval + citations + verifier |